In [3]:
import json

with open("school_codes_list.json", "r") as f:
    school_codes_list = json.load(f)

In [1]:
import pandas as pd
import requests
from io import StringIO

url = "https://www3.cde.ca.gov/demo-downloads/stability/sr2425-v2.txt"

r = requests.get(url)
r.raise_for_status()
text = r.content.decode("cp1252", errors="replace")

In [7]:
df = pd.read_csv(
    StringIO(text),
    sep="\t",
    dtype={
        "Academic Year": "string",
        "Aggregate Level": "string",
        "County Code": "string",
        "District Code": "string",
        "School Code": "string",
        "County Name": "string",
        "District Name": "string",
        "School Name": "string",
        "Charter School": "string",
        "DASS": "string",
        "Reporting Category": "string"
    },
    keep_default_na=False
)

df.columns = df.columns.str.strip()

string_cols = [
    "Academic Year", "Aggregate Level", "County Code", "District Code", "School Code",
    "County Name", "District Name", "School Name", "Charter School", "DASS",
    "Reporting Category"
]

for col in string_cols:
    df[col] = df[col].str.strip()

# keep only school-level total rows
stability_df = df[df["Aggregate Level"] == "S"].copy()
stability_df = stability_df[stability_df["Reporting Category"] == "TA"].copy()

# format codes
stability_df["School Code"] = stability_df["School Code"].astype(int).astype(str).str.zfill(7)

# filter to your schools
stability_df = stability_df[stability_df["School Code"].isin(school_codes_list)].copy()
stability_df = stability_df.reset_index(drop=True)

In [8]:
stability_df = stability_df[['School Code', 'Stability Rate (percent)']]

In [9]:
stability_df

,School Code,Stability Rate (percent)
0,0130625,72.6
1,0130229,96.7
2,0106401,97.7
3,0130450,97.2
4,0131177,92.9
...,...,...
1391,0113787,46.4
1392,5830013,81.3
1393,5835202,83.3
1394,5838305,88.4


In [ ]:
stability_df.to_csv('../cleaned_data/stability_data.csv')